[![Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giordamaug/decouple_emb_lgbm/blob/main/notebook.ipynb)

## Libraries

In [1]:
import torch
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action='ignore')
import numpy as np
from sklearn.metrics import (
    matthews_corrcoef, confusion_matrix, accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score
)
import lightgbm as lgb
import seaborn as sns
from tqdm.notebook import tqdm
import sys, os
import random
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"       # to force BERT determinsm

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # per multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

## Load dataset

In [2]:
import sys
sys.path.append('.')
from scripts.utils import Settings

datafile =  "data/ina_asplenic_1487.json" # "dataset_5flw_1inf.json" "mimic_patients_events.json"
args = Settings(datafile = datafile, 
                no_selection=False, 
                patologies=['Asplenic Syndromes', 
                            'Autoimmune Hematological Diseases', 
                            'Congenital Hemolytic Anemias', 
                            'Immunodeficiencies', 
                            'NTDT - (Non Transfusion Dependent Thalassemia - Intermediate Thalassemia)', 
                            'Non-Hemato-Oncological Causes', 
                            'Oncological Diseases', 
                            'SCD - (Sickle Cell Disease, Anemia/Sickle Cell or Sickle Cell Disease)', 
                            'TDT - (Transfusion Dependent Thalassemia - Thalassemia Major)'],
                methods=['LSTM', 'GRU', 'GRU-D', 'BEHRT', 'Dipole', 'DOME', 'BINARY'],
                evfields = ['event'],
                with_static=True,
                enable_plot=True, 
                n_splits=5, 
                num_epochs=10, 
                batch_size = 16, 
                embedding_size=128,
                hidden_size=300)

Error loading dataset: 'Settings' object has no attribute 'patologies_widget'


In [ ]:
datafile =  "data/ina_asplenic_1487.json"
dataset = pd.read_json(datafile)
len

In [7]:
dataset['base_pathology_area'].value_counts()

base_pathology_area
Congenital Hemolytic Anemias                                                 506
SCD - (Sickle Cell Disease, Anemia/Sickle Cell or Sickle Cell Disease)       402
TDT - (Transfusion Dependent Thalassemia - Thalassemia Major)                383
NTDT - (Non Transfusion Dependent Thalassemia - Intermediate Thalassemia)    181
Autoimmune Hematological Diseases                                            144
Non-Hemato-Oncological Causes                                                 90
Oncological Diseases                                                          60
Asplenic Syndromes                                                            13
Immunodeficiencies                                                            10
Name: count, dtype: int64

## Get clinical trajectories

In [ ]:
## Get sequences and static data
target = 'is_dead?'
removeevents = ['followup', 'platelet', 'bmi_change']
attributes = list(set(pd.read_csv("attributes.csv", header=None, comment = '#', index_col=0).index.to_list()).intersection(set(args.dataset.columns)))
'Static attributes' , attributes